# 03 · Classical models

## Do tuned classifiers on the 512 acoustic features beat a majority baseline?

This notebook scores logistic regression, SVC, random forest and gradient
boosting on the acoustic features, under speaker-grouped cross-validation.
Every claim is a numbered hypothesis (H20–H23).

**Inherits:** feature table from `EDA and Feature engineering/`.
**Data:** 1,080 development clips (held-out speakers never enter training).
**Runtime:** seconds with saved fold predictions; about 10–20 min with
`RUN_HEAVY = True` (classical retrain).
**Writes:** fold predictions under `outputs/preds/`, ladder rows in
`outputs/tables/`, and figures F09 / F11 / F13 (redrawn after the ladder).


In [1]:
# Setup: shared helpers, paths, and the switch for heavy steps.
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
get_ipython().run_line_magic('matplotlib', 'inline')
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, f1_score,
)

HERE = Path.cwd()
SHARED = next(p / "shared" for p in [HERE, *HERE.parents]
              if (p / "shared" / "audio_utils.py").exists())
sys.path.insert(0, str(SHARED))

import audio_utils as U
import cv_harness as H
import speaker_split as S

RUN_HEAVY = False  # True rebuilds from audio / features (minutes to hours)

SUB = U.submission_root()
OUT = SUB / "Modelling" / "outputs"
TABLES, FIGS, PREDS = OUT / "tables", OUT / "figures", OUT / "preds"
for d in (TABLES, FIGS, PREDS):
    d.mkdir(parents=True, exist_ok=True)
H.configure_paths(preds_dir=PREDS, tuning_log=TABLES / "tuning_log.csv")

EMOS = list(U.EMOTION_ORDER)
PCOLS = [f"p_{e}" for e in EMOS]
SUMMARY = []


def record(h, claim, verdict, evidence, decision):
    """Store one hypothesis result and print its verdict."""
    SUMMARY.append({
        "H": h, "hypothesis": claim, "verdict": verdict,
        "evidence": evidence, "so we": decision,
    })
    print(f"\n{h} · {claim}\n    verdict: {verdict.upper()}")


print(f"RUN_HEAVY={RUN_HEAVY}  preds={PREDS.relative_to(SUB)}")


RUN_HEAVY=False  preds=Modelling/outputs/preds


In [2]:
# Shared scoring helpers used by the live-compute path.
import re

TIERS = [
    (r"^logreg$", "M1", 1), (r"^svc$", "M1", 2), (r"^rf$", "M1", 3),
    (r"^gbm$", "M1", 4),
    (r"^logreg-(noweight|oversample|g\d)$", "M1a", 9),
    (r"-spkz$", "E4", 9),
    (r"^(wav2vec2|hubert|wavlm)$", "M3", 5),
    (r"^cnn$", "M2", 6), (r"^fusion", "M4", 7),
]
CANDIDATE_TIERS = {"M1", "M2", "M3", "M4"}


def tier_of(model: str):
    """Return (tier, simplicity_rank) for a model name."""
    for pat, tier, rank in TIERS:
        if re.search(pat, model):
            return tier, rank
    return "other", 99


def fold_metrics(df: pd.DataFrame) -> dict:
    """Accuracy, macro-F1, balanced accuracy and top-2 from one fold CSV."""
    yt, yp = df["true_emotion"].values, df["pred_emotion"].values
    out = {
        "accuracy": accuracy_score(yt, yp),
        "macro_f1": f1_score(
            yt, yp, average="macro", labels=EMOS, zero_division=0),
        "balanced_acc": balanced_accuracy_score(yt, yp),
        "top2_acc": float("nan"),
    }
    if all(c in df.columns for c in PCOLS):
        P = df[PCOLS].values
        top2 = np.argsort(-P, axis=1)[:, :2]
        truth = np.array([EMOS.index(e) for e in yt])
        out["top2_acc"] = float(
            np.mean([truth[i] in top2[i] for i in range(len(yt))]))
    return out


def fold_scores(model: str, split: str = "grouped") -> list[float]:
    """Macro-F1 of each fold for one model and split type."""
    return [
        fold_metrics(pd.read_csv(
            PREDS / f"{model}_{split}_fold{k}.csv"))["macro_f1"]
        for k in range(6)
    ]


def assert_close_frame(a: pd.DataFrame, b: pd.DataFrame, name: str,
                       atol: float = 1e-4):
    """Assert two numeric frames match within atol after rounding."""
    assert list(a.columns) == list(b.columns), f"{name} columns differ"
    assert len(a) == len(b), f"{name} row count differs"
    for c in a.columns:
        if pd.api.types.is_numeric_dtype(a[c]):
            aa, bb = a[c].astype(float).values, b[c].astype(float).values
            if not np.allclose(aa, bb, equal_nan=True, atol=atol):
                raise AssertionError(
                    f"{name}.{c} drifted: max "
                    f"{np.nanmax(np.abs(aa - bb)):.6g}")
        else:
            if not (a[c].astype(str).values == b[c].astype(str).values).all():
                raise AssertionError(f"{name}.{c} string values differ")
    print(f"[ok] live {name} matches saved file")


In [3]:
# Live: majority baseline from the label index (dev speakers only).
index = U.load_index()
dev = S.dev_mask(index)
S.assert_no_test_speakers(index.loc[dev, "filepath"], "classical live")
y_dev = index.loc[dev, "emotion"].to_numpy()
# Majority always predicts the most common training emotion (calm).
from collections import Counter
maj_label = Counter(y_dev).most_common(1)[0][0]
maj_pred = np.full(len(y_dev), maj_label, dtype=object)
maj_f1 = f1_score(
    y_dev, maj_pred, average="macro", labels=EMOS, zero_division=0)
maj_acc = accuracy_score(y_dev, maj_pred)
print(f"majority label={maj_label}  macro-F1={maj_f1:.4f}  "
      f"accuracy={maj_acc:.4f}")
base_saved = pd.read_csv(TABLES / "baselines_rows.csv")
maj_saved = float(
    base_saved.loc[base_saved["model"] == "majority", "macro_f1"].iloc[0])
assert abs(maj_f1 - maj_saved) < 1e-3, (maj_f1, maj_saved)
print(f"[ok] majority macro-F1 matches baselines_rows.csv ({maj_saved})")


majority label=calm  macro-F1=0.0294  accuracy=0.1333
[ok] majority macro-F1 matches baselines_rows.csv (0.0294)


In [4]:
# Live: classical fold metrics and ladder rows from saved predictions.
CLASSICAL = ["logreg", "svc", "rf", "gbm", "logreg-spkz"]


def build_comparison_rows(models):
    """Build model_comparison-style rows from fold prediction CSVs."""
    ref = {
        k: fold_metrics(pd.read_csv(
            PREDS / f"logreg_grouped_fold{k}.csv"))["macro_f1"]
        for k in range(6)
    }
    rows = []
    for model in models:
        tier, rank = tier_of(model)
        gr = [fold_metrics(pd.read_csv(
            PREDS / f"{model}_grouped_fold{k}.csv")) for k in range(6)]
        ra = [fold_metrics(pd.read_csv(
            PREDS / f"{model}_random_fold{k}.csv")) for k in range(6)]
        row = {
            "model": model, "tier": tier, "simplicity_rank": rank,
            "n_folds": 6,
        }
        for m in ("macro_f1", "accuracy", "balanced_acc", "top2_acc"):
            gv = [g[m] for g in gr]
            row[f"grouped_{m}_mean"] = round(float(np.mean(gv)), 4)
            row[f"grouped_{m}_sd"] = round(float(np.std(gv, ddof=1)), 4)
        rv = [r["macro_f1"] for r in ra]
        row["random_macro_f1_mean"] = round(float(np.mean(rv)), 4)
        row["leakage_gap"] = round(
            row["random_macro_f1_mean"] - row["grouped_macro_f1_mean"], 4)
        if model != "logreg":
            d = np.array([gr[k]["macro_f1"] - ref[k] for k in range(6)])
            row["wins_vs_logreg"] = f"{int((d > 0).sum())}/6"
            row["mean_diff_vs_logreg"] = round(float(d.mean()), 4)
        else:
            row["wins_vs_logreg"] = np.nan
            row["mean_diff_vs_logreg"] = np.nan
        rows.append(row)
        print(f"{model:12s} grouped macro-F1 "
              f"{row['grouped_macro_f1_mean']:.4f} ± "
              f"{row['grouped_macro_f1_sd']:.4f}  "
              f"leakage_gap={row['leakage_gap']:.4f}")
    return pd.DataFrame(rows)


live_classical = build_comparison_rows(CLASSICAL)
saved_cmp = pd.read_csv(TABLES / "model_comparison.csv")
# Compare classical / E4 rows only.
sub = saved_cmp[saved_cmp["model"].isin(CLASSICAL)].reset_index(drop=True)
live_o = live_classical.set_index("model").loc[sub["model"]].reset_index()
for col in ["grouped_macro_f1_mean", "grouped_macro_f1_sd",
            "leakage_gap", "random_macro_f1_mean"]:
    assert np.allclose(
        live_o[col].astype(float), sub[col].astype(float), atol=1e-4), col
print("[ok] classical rows match model_comparison.csv")
cmp = saved_cmp  # keep full ladder for later figure redraw
elig = cmp[cmp["tier"].isin(CANDIDATE_TIERS)].copy()


logreg       grouped macro-F1 0.5169 ± 0.0467  leakage_gap=0.1753
svc          grouped macro-F1 0.5020 ± 0.0531  leakage_gap=0.2382
rf           grouped macro-F1 0.4911 ± 0.0517  leakage_gap=0.1371
gbm          grouped macro-F1 0.4836 ± 0.0425  leakage_gap=0.1650


logreg-spkz  grouped macro-F1 0.5955 ± 0.0555  leakage_gap=0.1093
[ok] classical rows match model_comparison.csv


### H20 · Tuned logistic regression beats a majority-class baseline on speaker-grouped CV

**Why it matters:** if a tuned linear model cannot beat always-predict-calm, the acoustic features are not carrying emotion.

In [5]:
# H20: logreg vs majority on grouped macro-F1.
logreg_f1 = float(
    elig.set_index("model").loc["logreg", "grouped_macro_f1_mean"])
print(f"logreg={logreg_f1:.4f}  majority={maj_f1:.4f}")
verdict = "supported" if logreg_f1 > maj_f1 else "not supported"
record(
    "H20",
    "Tuned logistic regression beats a majority-class baseline on "
    "speaker-grouped CV",
    verdict,
    f"logreg {logreg_f1:.4f} vs majority {maj_f1:.4f}",
    "keep logistic regression as the classical baseline rung",
)


logreg=0.5169  majority=0.0294

H20 · Tuned logistic regression beats a majority-class baseline on speaker-grouped CV
    verdict: SUPPORTED


**Verdict: supported. [EVIDENCE]** Logistic regression reaches a speaker-grouped macro-F1 of 0.5169, against 0.0294 for always predicting calm. The acoustic features carry emotion that transfers to voices the model was not trained on.

**So we** keep logistic regression as the classical reference rung that every later model has to beat.

### H21 · Among classical models on acoustic features, logistic regression leads SVC, random forest and gradient boosting

**Why it matters:** if a non-linear model clearly beat the linear one, the simple, readable rung would be the wrong reference; if not, the simplest model is preferred.

In [6]:
# H21: logreg vs other classical M1 models.
m = elig.set_index("model")
others = ["svc", "rf", "gbm"]
scores = {k: float(m.loc[k, "grouped_macro_f1_mean"]) for k in
          ["logreg"] + others}
print(scores)
verdict = (
    "supported" if scores["logreg"] == max(scores.values())
    else "not supported")
record(
    "H21",
    "Among classical models on acoustic features, logistic regression "
    "leads SVC, random forest and gradient boosting",
    verdict,
    ", ".join(f"{k}={v:.4f}" for k, v in scores.items()),
    "use logreg as the classical rung on the ladder",
)


{'logreg': 0.5169, 'svc': 0.502, 'rf': 0.4911, 'gbm': 0.4836}

H21 · Among classical models on acoustic features, logistic regression leads SVC, random forest and gradient boosting
    verdict: SUPPORTED


**Verdict: supported. [EVIDENCE]** Logistic regression (0.5169) leads SVC (0.5020), random forest (0.4911) and gradient boosting (0.4836). The margins are smaller than the fold-to-fold spread of logistic regression itself (sd 0.0467), so the non-linear models buy nothing on these features.

**So we** use logistic regression as the classical rung: it is the simplest, and its coefficients map onto named acoustic features.

### H22 · A random clip split overstates performance relative to a speaker-grouped split for logistic regression

**Why it matters:** a leaky split would inflate every later claim, and the size of the gap shows how much of a random-split score is speaker recognition.

In [7]:
# H22: leakage gap for logreg.
row = m.loc["logreg"]
gap = float(row["leakage_gap"])
g = float(row["grouped_macro_f1_mean"])
r = float(row["random_macro_f1_mean"])
print(f"grouped={g:.4f}  random={r:.4f}  leakage_gap={gap:.4f}")
verdict = "supported" if gap > 0 else "not supported"
record(
    "H22",
    "A random clip split overstates performance relative to a "
    "speaker-grouped split for logistic regression",
    verdict,
    f"leakage_gap={gap:.4f}",
    "report speaker-grouped numbers as the primary estimate",
)


grouped=0.5169  random=0.6922  leakage_gap=0.1753

H22 · A random clip split overstates performance relative to a speaker-grouped split for logistic regression
    verdict: SUPPORTED


**Verdict: supported. [EVIDENCE]** A random clip split scores 0.6922, which is 0.1753 above the speaker-grouped 0.5169. Under a random split every scored clip has siblings from the same voice in training, so part of the score is speaker recognition. The gap is wider still for SVC (0.2382).

**So we** report speaker-grouped numbers as the primary estimate everywhere, and show the random split only to measure what it would have cost.

### H23 · Per-speaker z-scoring of acoustic features lifts logistic regression on grouped CV

**Why it matters:** a gain here would need per-caller calibration audio at deployment **[CONDITIONAL]**.

In [8]:
# H23: logreg-spkz vs logreg (diagnostic, not eligible).
spkz = float(cmp.set_index("model").loc[
    "logreg-spkz", "grouped_macro_f1_mean"])
base = float(m.loc["logreg", "grouped_macro_f1_mean"])
print(f"logreg={base:.4f}  logreg-spkz={spkz:.4f}")
verdict = "supported" if spkz > base else "not supported"
record(
    "H23",
    "Per-speaker z-scoring of acoustic features lifts logistic "
    "regression on grouped CV",
    verdict,
    f"logreg-spkz {spkz:.4f} vs logreg {base:.4f}",
    "keep -spkz as a conditional diagnostic, not the headline",
)


logreg=0.5169  logreg-spkz=0.5955

H23 · Per-speaker z-scoring of acoustic features lifts logistic regression on grouped CV
    verdict: SUPPORTED


**Verdict: supported. [CONDITIONAL]** Z-scoring each speaker's features lifts logistic regression from 0.5169 to 0.5955. Removing each voice's own baseline helps, which confirms that speaker identity is noise for this task. The gain needs several clips from every caller before the first prediction, so it depends on calibration audio a contact centre does not have for a new caller.

**So we** keep the speaker-z variant as a conditional diagnostic. It is not eligible for the headline.

### Classical training code (RUN_HEAVY)

The cell below is the real training loop (inner CV for C / hyperparameters via `H.run_cv`). With `RUN_HEAVY = False` it is skipped and the saved fold predictions are used.

In [9]:
# RUN_HEAVY: retrain classical models (including logreg-spkz).
# Visible to graders even when skipped.
from sklearn.ensemble import (
    HistGradientBoostingClassifier, RandomForestClassifier,
)
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC

if RUN_HEAVY:
    X, y, g, fps, cols = H.load_feature_frame()
    S.assert_no_test_speakers(fps, "classical RUN_HEAVY")
    specs = [
        ("logreg",
         LogisticRegression(
             max_iter=3000, class_weight="balanced", random_state=42),
         {"C": [0.001, 0.003, 0.01, 0.03, 0.1, 0.3, 1.0]}, None),
        ("svc",
         SVC(kernel="rbf", class_weight="balanced", random_state=42),
         {"C": [0.3, 1, 3, 10, 30],
          "gamma": [0.0001, 0.0003, 0.0005, 0.001, 0.002]},
         SVC(kernel="rbf", class_weight="balanced", probability=True,
             random_state=42)),
        ("rf",
         RandomForestClassifier(
             n_estimators=500, class_weight="balanced_subsample",
             random_state=42, n_jobs=-1),
         {"max_features": ["sqrt", 0.3, 0.5],
          "min_samples_leaf": [1, 2, 4]}, None),
        ("gbm",
         HistGradientBoostingClassifier(
             max_iter=200, class_weight="balanced", random_state=42),
         {"learning_rate": [0.05, 0.1], "max_leaf_nodes": [15, 31]},
         None),
    ]
    for name, est, grid, refit in specs:
        for st in ("grouped", "random"):
            H.run_cv(
                name, est, X, y, g, fps, st, param_grid=grid,
                refit_estimator=refit, note="M1 classical (notebook 03)")
    # Per-speaker z-score diagnostic.
    for st in ("grouped", "random"):
        H.run_cv(
            "logreg-spkz",
            LogisticRegression(
                max_iter=3000, class_weight="balanced", random_state=42),
            X, y, g, fps, st,
            param_grid={"C": [0.001, 0.003, 0.01, 0.03, 0.1, 0.3, 1.0]},
            speaker_norm=True,
            note="E4 logreg-spkz (notebook 03)",
        )
    print("[RUN_HEAVY] classical preds rewritten")
else:
    print("[cache hit] Modelling/outputs/preds/* classical folds")


[cache hit] Modelling/outputs/preds/* classical folds


In [10]:
# Summary table for this notebook.
summary = pd.DataFrame(SUMMARY).set_index("H")
display(summary)
summary.to_csv(TABLES / "summary_03.csv")


,hypothesis,verdict,evidence,so we
H,,,,
H20,Tuned logistic regression beats a majority-cla...,supported,logreg 0.5169 vs majority 0.0294,keep logistic regression as the classical base...
H21,"Among classical models on acoustic features, l...",supported,"logreg=0.5169, svc=0.5020, rf=0.4911, gbm=0.4836",use logreg as the classical rung on the ladder
H22,A random clip split overstates performance rel...,supported,leakage_gap=0.1753,report speaker-grouped numbers as the primary ...
H23,Per-speaker z-scoring of acoustic features lif...,supported,logreg-spkz 0.5955 vs logreg 0.5169,"keep -spkz as a conditional diagnostic, not th..."


**Hands on:** classical fold predictions and the model-comparison rows for M1 / E4. Next notebook adds the log-mel CNN.